In [79]:
import numpy as np
import pandas as pd

In [80]:
import os
import warnings
warnings.filterwarnings("ignore")

In [81]:
# Bayesian-smoothed rate. Prevents extreme values early in the season. Shrinks towards prior (league average) when sample size is small.

def smooth_rate(numerator, denominator, min_n=5, prior=0.1):
    return (numerator + prior * min_n) / (denominator + min_n)

In [82]:
# Three-level NaN fallback for circuit-specific features.
#     Level 1: actual computed value        (already in column)
#     Level 2: circuit average              (same circuit, all drivers)
#     Level 3: global feature mean          (entire dataset)
 
#     Used for circuit history features where a driver or team
#     has never visited the circuit before.

def circuit_nan_fill(df, feature):
    circuit_avg = df.groupby('circuit_id')[feature].transform('mean')
    df[feature] = df[feature].fillna(circuit_avg)
    df[feature] = df[feature].fillna(df[feature].mean())
    return df

In [83]:
# Fill NaN with average of all drivers in the same race round.
#     Used for rolling features where NaN means early season
#     (round 1 or 2) — other drivers in the same round have values.

def round_avg_fill(df, feature):
    round_avg = df.groupby(["season", "round"])[feature].transform('mean')
    df[feature] = df[feature].fillna(round_avg)
    return df

### Load data

In [84]:
race_raw = pd.read_csv("../data/raw/raw_race_data.csv")
historical_qual = pd.read_csv("../data/raw/raw_quali_data.csv")

In [85]:
race_raw.head()

,season,round,circuit_id,driver,team,finish_position,grid_position,points,status,dnf_flag,winner,circuit_type_encoded,circuit_avg_overtakes_last3years,circuit_safety_car_probability,circuit_lap_count,pit_stop_avg_time_loss_at_circuit
0,2022,1,Sakhir,LEC,Ferrari,1.0,1.0,26.0,Finished,0,1,0,18,0.45,NaN,21.5
1,2022,1,Sakhir,SAI,Ferrari,2.0,3.0,18.0,Finished,0,0,0,18,0.45,NaN,21.5
2,2022,1,Sakhir,HAM,Mercedes,3.0,5.0,15.0,Finished,0,0,0,18,0.45,NaN,21.5
3,2022,1,Sakhir,RUS,Mercedes,4.0,9.0,12.0,Finished,0,0,0,18,0.45,NaN,21.5
4,2022,1,Sakhir,MAG,Haas F1 Team,5.0,7.0,10.0,Finished,0,0,0,18,0.45,NaN,21.5


In [86]:
historical_qual.head()

,season,round,circuit_id,driver,team,quali_position,quali_best_lap_seconds,gap_to_pole_seconds,qualifying_session_was_dry
0,2022,1,Sakhir,LEC,Ferrari,1.0,90.558,0.000,1
1,2022,1,Sakhir,VER,Red Bull Racing,2.0,90.681,0.123,1
2,2022,1,Sakhir,SAI,Ferrari,3.0,90.687,0.129,1
3,2022,1,Sakhir,PER,Red Bull Racing,4.0,90.921,0.363,1
4,2022,1,Sakhir,HAM,Mercedes,5.0,91.048,0.490,1


In [87]:
# Sort chronologically - critical so shift(1) looks backward and not forward
race_raw = race_raw.sort_values(["season", "round", "driver"]).reset_index(drop=True)
historical_qual = historical_qual.sort_values(["season", "round", "driver"]).reset_index(drop=True)

In [88]:
# Merge qualifying position into race dataframe
historical_qual_slim = (
    historical_qual[["season", "round", "driver", "quali_position"]]
    .drop_duplicates(subset=["season", "round", "driver"])
)

df = race_raw.merge(historical_qual_slim, on=["season", "round", "driver"], how="left")

In [89]:
# Compute dynamic grid constants per race
# Handles both 20-driver (2022-2025) and 22-driver (2026+) grids
# Use finish_position count as proxy since grid_position not yet merged

actual_grid_size = (
    df.groupby(["season", "round"])["grid_position"]
    .transform("count")
)
actual_midfield = (actual_grid_size / 2).round()

## Feature Engineering

### BLOCK 1: TEAM / CAR PACE

In [90]:
# Rolling average team finish position
# Lower = better (1st place is best)

df["team_avg_finish_last_3"] = (
    df.groupby("team")["finish_position"]
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)
# NaN source: team's very first race ever (no history at all)
# Fix: fill with average of all teams in that round
df = round_avg_fill(df, "team_avg_finish_last_3")
df["team_avg_finish_last_3"] = df["team_avg_finish_last_3"].fillna(actual_midfield)
# Remaining NaN: global fallback

df["team_avg_finish_last_5"] = (
    df.groupby("team")["finish_position"].transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

df = round_avg_fill(df, "team_avg_finish_last_5")
df["team_avg_finish_last_5"] = df["team_avg_finish_last_5"].fillna(actual_midfield)

In [91]:
# Rolling points sum — captures consistency
df["team_points_last_3"] = (
    df.groupby("team")["points"].transform(lambda x: x.shift(1).rolling(3, min_periods=1).sum())
)

# NaN source: first race — no previous points
# Fix: 0 points is the correct semantic fill (earned nothing yet)
df["team_points_last_3"] = df["team_points_last_3"].fillna(0.0)

df["team_points_this_season"] = (
    df.groupby(["team", "season"])["points"].transform(lambda x: x.shift(1).expanding().sum())
).fillna(0.0)

# NaN source: round 1 before any points scored — 0 is correct

In [92]:
# Championship position — rank teams by cumulative points per race weekend
team_pts = (
    df.groupby(["season", "round", "team"])["team_points_this_season"]
    .first()
    .reset_index()
)
team_pts["team_championship_position"] = (
    team_pts
    .groupby(["season", "round"])["team_points_this_season"]
    .rank(ascending=False, method="min")
)
df = df.merge(
    team_pts[["season", "round", "team", "team_championship_position"]],
    on=["season", "round", "team"],
    how="left"
)

# NaN source: round 1 when all teams have 0 points (rank undefined)
# Fix: assume midfield position (5 out of ~10 teams)
df["team_championship_position"] = df["team_championship_position"].fillna(5.0)

In [93]:
# Points gap to championship leader — captures dominance
team_leader = (
    team_pts.groupby(["season", "round"])["team_points_this_season"].max().reset_index().rename(columns={"team_points_this_season": "_t1"})
)

In [94]:
df = df.merge(team_leader, on=["season", "round"], how="left")
df["team_points_gap_to_leader"] = df["_t1"] - df["team_points_this_season"]
df.drop(columns=["_t1"], inplace=True)

# NaN source: round 1 — no leader yet
# Fix: 0 gap (all teams equal at start)
df["team_points_gap_to_leader"] = df["team_points_gap_to_leader"].fillna(0.0)

In [95]:
# DNF rate — Bayesian smoothed with minimum 5 race denominator

df["_t_dnf"] = (
    df.groupby(["team", "season"])["dnf_flag"].transform(lambda x: x.shift(1).expanding().sum())
).fillna(0)

df["_t_races"] = (
    df.groupby(["team", "season"])["dnf_flag"].transform(lambda x: x.shift(1).expanding().count())
).fillna(0)

df["team_dnf_rate_this_season"] = smooth_rate(df["_t_dnf"], df["_t_races"])

# smooth_rate() never produces NaN — denominator always > 0 due to + min_n

In [96]:
# Finish rate (inverse of DNF rate — positive framing)
df["_t_fin"] = (
    df.groupby(["team", "season"])["dnf_flag"].transform(lambda x: (1 - x).shift(1).expanding().sum())
).fillna(0)

df["team_finish_rate_this_season"] = (
    df["_t_fin"] / df["_t_races"].replace(0, np.nan)
).fillna(0.90)

# NaN source: round 1 (zero races completed yet)
# Fix: 0.90 — league average finish rate as prior

### BLOCK 2: DRIVER FORM & SEASON PERFORMANCE

In [97]:
df["driver_avg_finish_last_3"] = (
    df.groupby("driver")["finish_position"]
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)

# NaN source: driver's very first race ever
# Fix: fill with round average (peers in same race), then midfield
df = round_avg_fill(df, "driver_avg_finish_last_3")
df["driver_avg_finish_last_3"] = df["driver_avg_finish_last_3"].fillna(actual_midfield)

df["driver_avg_finish_last_5"] = (
    df.groupby("driver")["finish_position"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

df = round_avg_fill(df, "driver_avg_finish_last_5")
df["driver_avg_finish_last_5"] = df["driver_avg_finish_last_5"].fillna(actual_midfield)
 
df["driver_points_this_season"] = (
    df.groupby(["driver", "season"])["points"]
    .transform(lambda x: x.shift(1).expanding().sum())
).fillna(0)

In [98]:
# Driver championship position
drv_pts = (
    df.groupby(["season", "round", "driver"])["driver_points_this_season"]
    .first()
    .reset_index()
)
drv_pts["driver_championship_position"] = (
    drv_pts
    .groupby(["season", "round"])["driver_points_this_season"]
    .rank(ascending=False, method="min")
)
df = df.merge(
    drv_pts[["season", "round", "driver", "driver_championship_position"]],
    on=["season", "round", "driver"],
    how="left"
)

# NaN source: round 1 all drivers on 0 points
# Fix: assume midfield (10 out of 20/22 drivers)
df["driver_championship_position"] = df["driver_championship_position"].fillna(actual_midfield)

In [99]:
# Points gap to championship leader
drv_leader = (
    drv_pts
    .groupby(["season", "round"])["driver_points_this_season"]
    .max()
    .reset_index()
    .rename(columns={"driver_points_this_season": "_dl"})
)
df = df.merge(drv_leader, on=["season", "round"], how="left")
df["driver_points_gap_to_leader"] = df["_dl"] - df["driver_points_this_season"]
df.drop(columns=["_dl"], inplace=True)

# NaN source: round 1
df["driver_points_gap_to_leader"] = df["driver_points_gap_to_leader"].fillna(0.0)

In [100]:
# DNF rate — Bayesian smoothed
df["_d_dnf"]   = (
    df.groupby(["driver", "season"])["dnf_flag"]
    .transform(lambda x: x.shift(1).expanding().sum())
).fillna(0)

df["_d_races"] = (
    df.groupby(["driver", "season"])["dnf_flag"]
    .transform(lambda x: x.shift(1).expanding().count())
).fillna(0)

df["driver_dnf_rate_this_season"] = smooth_rate(df["_d_dnf"], df["_d_races"])

# smooth_rate() never produces NaN

In [101]:
# Finish rate
df["_d_fin"] = (
    df.groupby(["driver", "season"])["dnf_flag"]
    .transform(lambda x: (1 - x).shift(1).expanding().sum())
).fillna(0)

df["driver_finish_rate_this_season"] = (
    df["_d_fin"] / df["_d_races"].replace(0, np.nan)
).fillna(0.90)

In [102]:
# Top-5 rate — more stable than wins or podiums
df["_top5_flag"] = (df["finish_position"] <= 5).astype(float)

df["_d_top5"]    = (
    df.groupby(["driver", "season"])["_top5_flag"]
    .transform(lambda x: x.shift(1).expanding().sum())
).fillna(0)

df["driver_top5_rate_this_season"] = (
    df["_d_top5"] / df["_d_races"].replace(0, np.nan)
).fillna(0.0)

# NaN source: round 1 — no races yet
# Fix: 0.0 — no top-5 finishes recorded (correct semantic)

### BLOCK 3: HISTORICAL QUALIFYING PERFORMANCE

In [103]:
df["driver_avg_quali_position_last_3"] = (
    df.groupby("driver")["quali_position"]
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
)

# NaN source: rookie first race, or quali data missing in FastF1
# Fix: round average first (peers in same race), then midfield
df = round_avg_fill(df, "driver_avg_quali_position_last_3")
df["driver_avg_quali_position_last_3"] = (
    df["driver_avg_quali_position_last_3"].fillna(actual_midfield)
)

df["driver_avg_quali_position_last_5"] = (
    df.groupby("driver")["quali_position"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

df = round_avg_fill(df, "driver_avg_quali_position_last_5")
df["driver_avg_quali_position_last_5"] = (
    df["driver_avg_quali_position_last_5"].fillna(actual_midfield)
)

df["driver_avg_quali_position_this_season"] = (
    df.groupby(["driver", "season"])["quali_position"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

df = round_avg_fill(df, "driver_avg_quali_position_this_season")
df["driver_avg_quali_position_this_season"] = (
    df["driver_avg_quali_position_this_season"].fillna(actual_midfield)
)

In [104]:
# Quali-to-finish delta
# Negative = driver gains places in race (great racer — Hamilton, Alonso type)
# Positive = driver loses places in race (quali specialist)
df["_q2f"] = df["finish_position"] - df["quali_position"]
df["driver_quali_to_finish_delta_last_5"] = (
    df.groupby("driver")["_q2f"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

# NaN source: rookie first race, or quali data missing
# Fix: 0 — neutral assumption (finishes where they qualify)
df["driver_quali_to_finish_delta_last_5"] = (
    df["driver_quali_to_finish_delta_last_5"].fillna(0.0)
)

### BLOCK 4: TEAMMATE COMPARISON

In [105]:
# Season average qualifying position per driver before this race
df["_drv_s_quali"] = (
    df.groupby(["driver", "season"])["quali_position"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

# Team mean of season avg quali — gap = driver minus team mean
df["_team_s_quali"] = (
    df.groupby(["season", "round", "team"])["_drv_s_quali"]
    .transform("mean")
)

# Negative = driver qualifies ahead of teammates on average
df["avg_quali_gap_to_teammate_this_season"] = (
    df["_drv_s_quali"] - df["_team_s_quali"]
)

# NaN source: round 1, single-driver team, quali data missing
# Fix: 0 — neutral (neither ahead nor behind teammate)
df["avg_quali_gap_to_teammate_this_season"] = (
    df["avg_quali_gap_to_teammate_this_season"].fillna(0.0)
)
 
# Season average finish position per driver
df["_drv_s_fin"] = (
    df.groupby(["driver", "season"])["finish_position"]
    .transform(lambda x: x.shift(1).expanding().mean())
)

df["_team_s_fin"] = (
    df.groupby(["season", "round", "team"])["_drv_s_fin"]
    .transform("mean")
)

# Negative = driver finishes ahead of teammates on average
df["avg_finish_gap_to_teammate_this_season"] = (
    df["_drv_s_fin"] - df["_team_s_fin"]
)

# NaN source: round 1, single-driver team
# Fix: 0 — neutral
df["avg_finish_gap_to_teammate_this_season"] = (
    df["avg_finish_gap_to_teammate_this_season"].fillna(0.0)
)

In [106]:
# Head-to-head ratio: proportion of races finished ahead of teammate
h2h_rows = []
for (season, rnd, team), grp in df.groupby(["season", "round", "team"]):
    valid = grp[["driver", "finish_position"]].dropna()
    if len(valid) < 2:
        continue
    for _, r in valid.iterrows():
        beat = (valid["finish_position"] > r["finish_position"]).sum()
        h2h_rows.append({
            "season": season,
            "round":  rnd,
            "driver": r["driver"],
            "_beat":  1 if beat > 0 else 0,
        })
 
if h2h_rows:
    h2h_df = pd.DataFrame(h2h_rows).sort_values(["season", "round"])
    h2h_df["teammate_head_to_head_ratio_this_season"] = (
        h2h_df.groupby(["driver", "season"])["_beat"]
        .transform(lambda x: x.shift(1).expanding().mean())
    )
    df = df.merge(
        h2h_df[["season", "round", "driver",
                 "teammate_head_to_head_ratio_this_season"]],
        on=["season", "round", "driver"],
        how="left"
    )
else:
    df["teammate_head_to_head_ratio_this_season"] = np.nan

In [107]:
# Default 0.5 for first race or single-driver teams
# NaN source: round 1, single-driver team, DNF before teammate
# Fix: 0.5 — neutral (50/50 unknown)
df["teammate_head_to_head_ratio_this_season"] = (
    df["teammate_head_to_head_ratio_this_season"].fillna(0.5)
)

### BLOCK 5: DRIVER CIRCUIT HISTORY

In [108]:
df["driver_avg_finish_at_circuit_last5"] = (
    df.groupby(["driver", "circuit_id"])["finish_position"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)
# NaN source: driver has never visited this circuit before
# Fix: circuit_nan_fill — circuit avg -> global avg
df = circuit_nan_fill(df, "driver_avg_finish_at_circuit_last5")

df["driver_best_finish_at_circuit_last5"] = (
    df.groupby(["driver", "circuit_id"])["finish_position"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).min())
)
df = circuit_nan_fill(df, "driver_best_finish_at_circuit_last5")
 
# NaN source: never visited circuit
# Fix: 0 — no podiums recorded is the correct semantic
df["_pod_flag"] = (df["finish_position"] <= 3).astype(float)
df["driver_podiums_at_circuit_last5"] = (
    df.groupby(["driver", "circuit_id"])["_pod_flag"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
).fillna(0)
 
df["driver_avg_quali_position_at_circuit_last5"] = (
    df.groupby(["driver", "circuit_id"])["quali_position"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)
df = circuit_nan_fill(df, "driver_avg_quali_position_at_circuit_last5")
 
df["driver_dnf_rate_at_circuit"] = (
    df.groupby(["driver", "circuit_id"])["dnf_flag"]
    .transform(lambda x: x.shift(1).expanding().mean())
)
df = circuit_nan_fill(df, "driver_dnf_rate_at_circuit")
 
# Total visits before this race — experience proxy
# NaN source: first ever visit
# Fix: 0 — correct semantic (zero prior visits)
df["driver_visits_to_circuit"] = (
    df.groupby(["driver", "circuit_id"])["finish_position"]
    .transform(lambda x: x.shift(1).expanding().count())
).fillna(0)

In [109]:
# Circuit specialist signal
# Negative = driver outperforms their general average at this circuit
# e.g. Alonso at Monaco shows strongly negative value
df["driver_circuit_vs_season_delta"] = (
    df["driver_avg_finish_at_circuit_last5"]
    - df["driver_avg_finish_last_5"]
)

# NaN source: both component features already filled — should be NaN-free
# Safety net: fill with 0 (neutral — no known specialist advantage)
df["driver_circuit_vs_season_delta"] = (
    df["driver_circuit_vs_season_delta"].fillna(0.0)
)

### BLOCK 6: TEAM CIRCUIT HISTORY

In [110]:
df["team_avg_finish_at_circuit_last5"] = (
    df.groupby(["team", "circuit_id"])["finish_position"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)
df = circuit_nan_fill(df, "team_avg_finish_at_circuit_last5")
 

df["team_wins_at_circuit_last5"] = (
    df.groupby(["team", "circuit_id"])["winner"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).sum())
)
# NaN source: team never visited circuit
# Fix: 0 — no wins recorded is correct semantic
df["team_wins_at_circuit_last5"] = df["team_wins_at_circuit_last5"].fillna(0.0)
 
df["_tpod_flag"] = (df["finish_position"] <= 3).astype(float)
df["team_podium_rate_at_circuit"] = (
    df.groupby(["team", "circuit_id"])["_tpod_flag"]
    .transform(lambda x: x.shift(1).expanding().mean())
)
df = circuit_nan_fill(df, "team_podium_rate_at_circuit")
 
df["team_avg_quali_position_at_circuit_last5"] = (
    df.groupby(["team", "circuit_id"])["quali_position"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)
df = circuit_nan_fill(df, "team_avg_quali_position_at_circuit_last5")
 
df["team_dnf_rate_at_circuit"] = (
    df.groupby(["team", "circuit_id"])["dnf_flag"]
    .transform(lambda x: x.shift(1).expanding().mean())
)
df = circuit_nan_fill(df, "team_dnf_rate_at_circuit")

In [111]:
# Car-circuit fit signal
# Negative = car concept is particularly strong at this circuit layout
# e.g. Red Bull at Baku, Mercedes at Barcelona historically
df["team_circuit_vs_season_delta"] = (
    df["team_avg_finish_at_circuit_last5"]
    - df["team_avg_finish_last_5"]
)

# Safety net: 0 = neutral (no known car-circuit fit advantage)
df["team_circuit_vs_season_delta"] = (
    df["team_circuit_vs_season_delta"].fillna(0.0)
)

### BLOCK 7: CIRCUIT CONTEXT

In [112]:
# circuit_type_encoded, circuit_avg_overtakes_last3years,
# circuit_safety_car_probability already extracted in extractor.py
# Only circuit_lap_count needs cleaning
df["circuit_type_encoded"] = (
    df["circuit_type_encoded"].fillna(0)  # assume permanent circuit
)
df["circuit_avg_overtakes_last3years"] = (
    df["circuit_avg_overtakes_last3years"].fillna(18)  # global average
)
df["circuit_safety_car_probability"] = (
    df["circuit_safety_car_probability"].fillna(0.45)  # global average
)


df["circuit_lap_count"] = pd.to_numeric(df["circuit_lap_count"], errors="coerce")
lap_med = df.groupby("circuit_id")["circuit_lap_count"].transform("median")
df["circuit_lap_count"] = df["circuit_lap_count"].fillna(lap_med).fillna(57)

### BLOCK 8: SEASON & ROUND CONTEXT

In [113]:

df["season_round_number"] = df["round"]
 
df["races_remaining_this_season"] = (
    df.groupby("season")["round"].transform("max") - df["round"]
)
 
df["is_season_opener"] = (df["round"] == 1).astype(int)

In [114]:
# Captures how settled the regulation era is
# 2022 = 0 (brand new regs, high uncertainty)
# 2025 = 3 (settled, patterns more stable)
df["seasons_since_regulation_change"] = df["season"] - 2022

### DROP INTERMEDIATE COLUMNS

In [115]:
intermediate_cols = [
    "_t_dnf", "_t_races", "_t_fin",
    "_d_dnf", "_d_races", "_d_fin",
    "_d_top5", "_top5_flag",
    "_q2f",
    "_drv_s_quali", "_team_s_quali",
    "_drv_s_fin",   "_team_s_fin",
    "_beat",
    "_pod_flag", "_tpod_flag",
    "status", "dnf_flag", "points",
    "finish_position", "quali_position", "grid_position"
]

df.drop(columns=[c for c in intermediate_cols if c in df.columns], inplace=True)

### DEFINE FINAL COLUMN ORDER AND SAVE

In [ ]:
identity_cols = ["season", "round", "circuit_id", "driver", "team"]
 
pre_qual_features = [
    # Block 1 — Team Car Pace (8)
    "team_avg_finish_last_3",
    "team_avg_finish_last_5",
    "team_points_last_3",
    "team_points_this_season",
    "team_championship_position",
    "team_points_gap_to_leader",
    "team_dnf_rate_this_season",
    "team_finish_rate_this_season",
    # Block 2 — Driver Form (8)
    "driver_avg_finish_last_3",
    "driver_avg_finish_last_5",
    "driver_points_this_season",
    "driver_championship_position",
    "driver_points_gap_to_leader",
    "driver_dnf_rate_this_season",
    "driver_finish_rate_this_season",
    "driver_top5_rate_this_season",
    # Block 3 — Historical Qualifying (4)
    "driver_avg_quali_position_last_3",
    "driver_avg_quali_position_last_5",
    "driver_avg_quali_position_this_season",
    "driver_quali_to_finish_delta_last_5",
    # Block 4 — Teammate Comparison (3)
    "avg_quali_gap_to_teammate_this_season",
    "avg_finish_gap_to_teammate_this_season",
    "teammate_head_to_head_ratio_this_season",
    # Block 5 — Driver Circuit History (7)
    "driver_avg_finish_at_circuit_last5",
    "driver_best_finish_at_circuit_last5",
    "driver_podiums_at_circuit_last5",
    "driver_avg_quali_position_at_circuit_last5",
    "driver_dnf_rate_at_circuit",
    "driver_visits_to_circuit",
    "driver_circuit_vs_season_delta",
    # Block 6 — Team Circuit History (6)
    "team_avg_finish_at_circuit_last5",
    "team_wins_at_circuit_last5",
    "team_podium_rate_at_circuit",
    "team_avg_quali_position_at_circuit_last5",
    "team_dnf_rate_at_circuit",
    "team_circuit_vs_season_delta",
    # Block 7 — Circuit Context (4)
    "circuit_type_encoded",
    "circuit_avg_overtakes_last3years",
    "circuit_safety_car_probability",
    "circuit_lap_count",
    # Block 8 — Season Round Context (4)
    "season_round_number",
    "races_remaining_this_season",
    "is_season_opener",
    "seasons_since_regulation_change",
]
 
TARGET = "winner"
 
# Validate all expected features are present
available = [c for c in pre_qual_features if c in df.columns]
missing   = [c for c in pre_qual_features if c not in df.columns]
if missing:
    print(f"\n Missing features: {missing}")
 
final_cols = identity_cols + available + [TARGET]
df_out     = df[[c for c in final_cols if c in df.columns]].copy()
 
df_out.to_csv("../data/processed/pre_qualifying_dataset.csv", index=False)

In [117]:
df.shape

(1838, 51)